In [0]:
%sql
CREATE OR REPLACE TABLE aml_risk_prediction.bronze.bronze_table AS
SELECT 
  `Timestamp` AS timestamp, 
  `From Bank` AS from_bank, 
  `Account` AS account,
  `To Bank` AS to_bank,
  `Amount Received` AS amount_received,
  `Receiving Currency` AS receiving_currency, 
  `Amount Paid` AS amount_paid, 
  `Payment Currency` AS payment_currency, 
  `Payment Format` AS payment_format, 
  `Is Laundering` AS is_laundering,
  current_date() AS ingestion_time
FROM aml_risk_prediction.bronze.bronze_table;

In [0]:
%sql
CREATE OR REPLACE TABLE aml_risk_prediction.bronze.bronze_table2 AS 
SELECT 
  `Bank Name` AS bank_name, 
  `Bank ID` AS bank_id, 
  `Account Number` AS account_number, 
  `Entity ID` AS entity_id, 
  `Entity Name` AS entity_name
FROM aml_risk_prediction.bronze.bronze_table2;


In [0]:
%sql
SELECT * FROM aml_risk_prediction.bronze.bronze_table LIMIT 10

In [0]:
import pandas as pd
import numpy as np

# Read your Bronze transaction data
df = spark.table("aml_risk_prediction.bronze.bronze_table").toPandas()

print("Rows:", len(df))
print(df.columns.tolist())

In [0]:
# ============================================
# COMPREHENSIVE DATA CLEANING FOR SILVER LAYER
# ============================================

# 1. HANDLE MISSING VALUES
# Drop rows with missing critical fields
critical_fields = ['timestamp', 'account', 'amount_received', 'amount_paid']
df_cleaned = df.dropna(subset=critical_fields)

# Fill missing categorical values with 'UNKNOWN'
categorical_cols = ['from_bank', 'to_bank', 'receiving_currency', 'payment_currency', 'payment_format']
for col in categorical_cols:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].fillna('UNKNOWN')

# 2. DATA TYPE CONVERSION & VALIDATION
# Convert amount fields to numeric (coerce errors to NaN)
df_cleaned['amount_received'] = pd.to_numeric(df_cleaned['amount_received'], errors='coerce')
df_cleaned['amount_paid'] = pd.to_numeric(df_cleaned['amount_paid'], errors='coerce')

# Ensure timestamp is datetime
df_cleaned['timestamp'] = pd.to_datetime(df_cleaned['timestamp'], errors='coerce')

# Convert is_laundering to boolean for clarity
df_cleaned['is_laundering'] = df_cleaned['is_laundering'].astype(bool)

# Remove any rows where numeric conversions failed
df_cleaned = df_cleaned.dropna(subset=['amount_received', 'amount_paid', 'timestamp'])

# 3. HANDLE NEGATIVE AND ZERO VALUES
# Remove transactions with negative or zero amounts
df_cleaned = df_cleaned[
    (df_cleaned['amount_received'] > 0) & 
    (df_cleaned['amount_paid'] > 0)
]

# 4. OUTLIER DETECTION AND HANDLING
# Calculate IQR for amount_received
Q1_received = df_cleaned['amount_received'].quantile(0.25)
Q3_received = df_cleaned['amount_received'].quantile(0.75)
IQR_received = Q3_received - Q1_received
lower_bound_received = Q1_received - 3 * IQR_received
upper_bound_received = Q3_received + 3 * IQR_received

# Calculate IQR for amount_paid
Q1_paid = df_cleaned['amount_paid'].quantile(0.25)
Q3_paid = df_cleaned['amount_paid'].quantile(0.75)
IQR_paid = Q3_paid - Q1_paid
lower_bound_paid = Q1_paid - 3 * IQR_paid
upper_bound_paid = Q3_paid + 3 * IQR_paid

# Flag outliers (keep them but mark for analysis)
df_cleaned['is_outlier_amount'] = (
    (df_cleaned['amount_received'] < lower_bound_received) |
    (df_cleaned['amount_received'] > upper_bound_received) |
    (df_cleaned['amount_paid'] < lower_bound_paid) |
    (df_cleaned['amount_paid'] > upper_bound_paid)
)

# 5. REMOVE EXACT DUPLICATES
df_cleaned = df_cleaned.drop_duplicates(
    subset=['timestamp', 'account', 'from_bank', 'to_bank', 'amount_received'],
    keep='first'
)

# 6. DATA VALIDATION AND BUSINESS RULES
# Ensure account is string and non-empty
df_cleaned['account'] = df_cleaned['account'].astype(str).str.strip()
df_cleaned = df_cleaned[df_cleaned['account'].str.len() > 0]

# Standardize currency codes (uppercase)
df_cleaned['receiving_currency'] = df_cleaned['receiving_currency'].str.upper().str.strip()
df_cleaned['payment_currency'] = df_cleaned['payment_currency'].str.upper().str.strip()

# Standardize payment format
df_cleaned['payment_format'] = df_cleaned['payment_format'].str.upper().str.strip()

In [0]:
df_cleaned.info()

In [0]:
df_cleaned.head()

In [0]:
# Convert back to Spark DataFrame
spark_df_silver = spark.createDataFrame(df_cleaned)

# Save as Silver Delta Table (Overwrite or Append based on need)
spark_df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aml_risk_prediction.silver.silver_transactions")

print("Successfully written to aml_risk_prediction.silver.silver_transactions")

In [0]:
# Convert back to Spark DataFrame
spark_df_silver = spark.createDataFrame(df_cleaned)

# Save as Silver Delta Table (Overwrite or Append based on need)
spark_df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aml_risk_prediction.silver.silver_transactions")

print("Successfully written to aml_risk_prediction.silver.silver_transactions")

In [0]:
%sql
-- 1. Check summary counts and outlier distribution
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN is_outlier_amount THEN 1 ELSE 0 END) AS outlier_count,
    SUM(CASE WHEN is_laundering THEN 1 ELSE 0 END) AS laundering_count
FROM aml_risk_prediction.silver.silver_transactions;

In [0]:
%sql
SELECT * FROM aml_risk_prediction.bronze.bronze_table2 LIMIT 10;

In [0]:
df_bronze2 = spark.table("aml_risk_prediction.bronze.bronze_table2")

# 2. Write directly as Silver Delta Table
df_bronze2.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("aml_risk_prediction.silver.silver_table2")

print("Successfully created aml_risk_prediction.silver.silver_table2")